In [ ]:
# Cell 1
# Mục đích: khai báo lại client và các hàm đo (notebook mới phải khai báo lại).
from google.cloud import bigquery

PROJECT = "bq-learning-510104"
client = bigquery.Client(project=PROJECT, location="US")   # kênh kết nối BigQuery, vùng US

def mb(n):
    # đổi byte sang MB; trả None nếu n là None để lỗi "đọc byte quá sớm" lộ ra
    return None if n is None else round(n / 1024 / 1024, 2)

def check(sql):
    # dry run: chỉ ước lượng byte sẽ quét, không chạy thật
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    job = client.query(sql, job_config=cfg)
    print(f"Sẽ quét: {mb(job.total_bytes_processed)} MB")

def stats(sql, max_mb=100):
    # chạy thật; max_mb là tham số mặc định; trần so với bytes BILLED
    cfg = bigquery.QueryJobConfig(
        use_query_cache=False,
        maximum_bytes_billed=max_mb * 1024 * 1024,   # đổi MB sang byte
    )
    job = client.query(sql, job_config=cfg)          # gửi job, chưa chờ
    df = job.result().to_dataframe()                 # chờ xong rồi đưa kết quả vào df
    print(f"processed: {mb(job.total_bytes_processed)} MB | billed: {mb(job.total_bytes_billed)} MB")
    return df

In [21]:
# Cell 2
# Mục đích: đo dung lượng file .pbix (chạy lần 1 trước khi sửa, lần 2 sau khi sửa).
from pathlib import Path

PBIX = Path(r"D:\BI\W4D4_backup.pbix")   # SỬA: tên file .pbix thật của bạn

def pbix_size():
    if not PBIX.exists():                      # sai đường dẫn thì báo, không đoán
        print("Không thấy file, kiểm tra lại đường dẫn")
        return
    size = PBIX.stat().st_size                 # kích thước file, đơn vị byte
    print(f"{PBIX.name}: {size:,} bytes = {size / 1024 / 1024:.2f} MB")

pbix_size()

W4D4_backup.pbix: 1,540,766 bytes = 1.47 MB


In [ ]:
# Cell 3
# Mục đích: lấy số KPI gốc bằng SQL độc lập để đối chiếu với Power BI trước và sau khi sửa.
# Chi phí: đọc 1 bảng, billed khoảng 10 MB.
sql = """
SELECT
  COUNT(*)                                  AS order_items,
  COUNT(DISTINCT order_id)                  AS orders,
  ROUND(SUM(sale_price), 2)                 AS net_revenue,
  ROUND(SUM(IF(EXTRACT(YEAR FROM order_date) = 2026, sale_price, 0)), 2) AS net_revenue_2026,
  MAX(order_date)                           AS data_through
FROM `bq-learning-510104.dbt_dev.fct_net_sales`
"""
check(sql)            # xem byte trước
df = stats(sql)
print(len(df))        # mong đợi 1
df.T                  # xoay cho dễ đọc

Sẽ quét: 1.04 MB
processed: 1.04 MB | billed: 10.0 MB
1


,0
order_items,45285
orders,31203
net_revenue,2707334.88
f0_,982813.17
data_through,2026-10-22


In [17]:
# Cell 4
# Mục đích: đếm số giá trị khác nhau của từng cột fact để biết cột nào "nặng" và cột nào Power BI không dùng.
# Chi phí: đọc 1 bảng (cả 8 cột), billed khoảng 10 MB.
sql = """
SELECT
  COUNT(*)                         AS so_dong,
  COUNT(DISTINCT order_item_id)    AS d_order_item_id,
  COUNT(DISTINCT order_id)         AS d_order_id,
  COUNT(DISTINCT user_id)          AS d_user_id,
  COUNT(DISTINCT product_id)       AS d_product_id,
  COUNT(DISTINCT order_date)       AS d_order_date,
  COUNT(DISTINCT sale_price)       AS d_sale_price,
  COUNT(DISTINCT cost)             AS d_cost,
  COUNT(DISTINCT gross_profit)     AS d_gross_profit
FROM `bq-learning-510104.dbt_dev.fct_net_sales`
"""
check(sql)
df = stats(sql)
print(len(df))
df.T

Sẽ quét: 2.76 MB
processed: 2.76 MB | billed: 10.0 MB
1


,0
so_dong,45285
d_order_item_id,45285
d_order_id,31203
d_user_id,27546
d_product_id,22840
d_order_date,2670
d_sale_price,3681
d_cost,21071
d_gross_profit,21079


In [22]:
# Cell 5
# Mục đích: xem lần Refresh (Close & Apply) vừa rồi Power BI gửi gì lên BigQuery.
# Chi phí: đọc INFORMATION_SCHEMA.JOBS, thường billed khoảng 20 MB.
sql = """
SELECT
  creation_time,
  statement_type,
  ROUND(total_bytes_processed / 1024 / 1024, 2) AS processed_mb,
  ROUND(total_bytes_billed / 1024 / 1024, 2)    AS billed_mb,
  SUBSTR(query, 1, 200)                         AS query_head
FROM `bq-learning-510104.region-us.INFORMATION_SCHEMA.JOBS`
WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 24 HOUR)
  AND job_type = 'QUERY'
  AND query NOT LIKE '/* {"app": "dbt"%'
  AND query LIKE '%dbt_dev%'
  AND NOT REGEXP_CONTAINS(query, r'INFORMATION_SCHEMA|d_order_item_id|net_revenue_2026')
ORDER BY creation_time DESC
LIMIT 30
"""
check(sql)
df = stats(sql)
print(len(df))                      # bảng rỗng thì vòng for bên dưới chạy 0 lần
for _, row in df.iterrows():        # hai tên biến: _ nhận số thứ tự, row nhận cả dòng
    print(row["creation_time"], row["processed_mb"], row["billed_mb"])
    print(row["query_head"])
    print("-" * 60)

Sẽ quét: 0.01 MB
processed: 0.01 MB | billed: 20.0 MB
19
2026-10-22 07:21:06.061000+00:00 1.04 10.0

SELECT
  COUNT(*) AS order_items
  ,COUNT(DISTINCT order_id) as orders
  ,ROUND(SUM(sale_price),2) as net_revenue
  ,ROUND(SUM(IF(EXTRACT(YEAR FROM order_date) =2026, sale_price, 0)),2)
  ,MAX(order_
------------------------------------------------------------
2026-10-22 07:21:04.861000+00:00 1.04 10.0

SELECT
  COUNT(*) AS order_items
  ,COUNT(DISTINCT order_id) as orders
  ,ROUND(SUM(sale_price),2) as net_revenue
  ,ROUND(SUM(IF(EXTRACT(YEAR FROM order_date) =2026, sale_price, 0)),2)
  ,MAX(order_
------------------------------------------------------------
2026-10-22 07:19:06.582000+00:00 nan nan

SELECT
  COUNT(*) AS order_items
  ,COUNT(DISTINCT order_id) as orders
  ,ROUND(SUM(sale_price),2) as net_revenue
  ,ROUND(SUM(IF(EXTRACT(YEAR FROM order_date) =2026, sale_price, 0), 2 ))
  ,MAX(orde
------------------------------------------------------------
2026-10-22 07:19:05.904000+00:

In [23]:
# Cell 5b
# Mục đích: lọc lại danh sách job, bỏ truy vấn của notebook, chỉ giữ job nghi là của Power BI.
# Chi phí: đọc INFORMATION_SCHEMA.JOBS, thường billed khoảng 20 MB.
sql = r"""
SELECT
  creation_time,
  statement_type,
  ROUND(total_bytes_processed / 1024 / 1024, 2) AS processed_mb,
  ROUND(total_bytes_billed / 1024 / 1024, 2)    AS billed_mb,
  SUBSTR(REGEXP_REPLACE(query, r'\s+', ' '), 1, 250) AS query_head
FROM `bq-learning-510104.region-us.INFORMATION_SCHEMA.JOBS`
WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 24 HOUR)
  AND job_type = 'QUERY'
  AND query NOT LIKE '/* {"app": "dbt"%'
  AND query LIKE '%dbt_dev%'
  AND query NOT LIKE '%COUNT(%'
  AND query NOT LIKE '%INFORMATION_SCHEMA%'
ORDER BY creation_time DESC
LIMIT 30
"""
check(sql)
df = stats(sql)
print(len(df))
for _, row in df.iterrows():
    print(row["creation_time"], row["processed_mb"], row["billed_mb"])
    print(row["query_head"])
    print("-" * 60)

Sẽ quét: 0.01 MB
processed: 0.01 MB | billed: 20.0 MB
0


In [24]:
# Cell 5c
# Mục đích: xem MỌI job trong 60 phút qua (không lọc theo tên bảng) để biết Power BI có tạo job không.
# Chi phí: đọc INFORMATION_SCHEMA.JOBS, thường billed khoảng 20 MB.
sql = r"""
SELECT
  creation_time,
  job_type,
  statement_type,
  ROUND(total_bytes_billed / 1024 / 1024, 2) AS billed_mb,
  ARRAY_TO_STRING(ARRAY(SELECT t.table_id FROM UNNEST(referenced_tables) AS t), ', ') AS tables_read,
  SUBSTR(REGEXP_REPLACE(query, r'\s+', ' '), 1, 150) AS query_head
FROM `bq-learning-510104.region-us.INFORMATION_SCHEMA.JOBS`
WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 60 MINUTE)
  AND (query IS NULL OR query NOT LIKE '%INFORMATION_SCHEMA%')
ORDER BY creation_time DESC
LIMIT 40
"""
check(sql)
df = stats(sql)
print(len(df))
for _, row in df.iterrows():
    print(row["creation_time"], row["job_type"], row["statement_type"], row["billed_mb"])
    print("bảng:", row["tables_read"])
    print(row["query_head"])
    print("-" * 60)

Sẽ quét: None MB
processed: 0.0 MB | billed: 0.0 MB
0
